# Literature Database Builder With Source Registry

This notebook is a source-transparent literature database builder.

It is designed to make the workflow auditable, so you can clearly show:

1. which papers and datasets were included
2. which URLs or PDFs were used as inputs
3. which structured literature records were extracted
4. which formatted references should be cited in presentations or reports


## Workflow Visibility Goal

This notebook keeps a clean separation between:

- source registration
- database building
- paper visibility and provenance review
- formatted reference export

The aim is that the top of the workflow should always make it easy to answer:

- What literature is currently included?
- What is each paper used for?
- What title / DOI / paper ID corresponds to each extracted evidence record?


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Markdown

project_dir = Path.cwd()
sys.path.append(str(project_dir))

from neurodesk_literature_to_pgsql import (
    analyze_sources,
    build_evidence_framework,
    build_pathology_evidence_table,
    build_roi_evidence_table,
)


## Stage 1. Register Literature Inputs

The source registry is now split into two folders instead of one empty/general JSON file:

- `workflow_sources/Disease/disease_literature_sources.json`
  - disease/pathology/subtype evidence
  - cohort and dataset provenance sources
- `workflow_sources/ROI/roi_literature_sources.json`
  - ROI, MRI/PET, imaging-methods, and regional-feature evidence sources

Each source row should include:

- `paper_code`: short ID used in evidence tables
- `source_label`: short human-readable label
- `input_value`: URL or local PDF path
- `evidence_category`: `disease_evidence` or `roi_evidence`
- `disease_focus`: main disease or syndrome supported
- `note`: why the source is included and how it should be interpreted


In [ ]:
SOURCE_ROOT = Path('./workflow_sources')
DISEASE_SOURCE_JSON_PATH = SOURCE_ROOT / 'Disease' / 'disease_literature_sources.json'
ROI_SOURCE_JSON_PATH = SOURCE_ROOT / 'ROI' / 'roi_literature_sources.json'
SOURCE_REGISTRY_MANIFEST_PATH = SOURCE_ROOT / 'literature_source_registry.json'


def load_source_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Source registry JSON not found: {path}')
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list):
        raise ValueError(f'Source registry must be a list of records: {path}')
    return data


DISEASE_SOURCES = load_source_json(DISEASE_SOURCE_JSON_PATH)
ROI_SOURCES = load_source_json(ROI_SOURCE_JSON_PATH)
LITERATURE_SOURCES = DISEASE_SOURCES + ROI_SOURCES

OUTPUT_DIR = Path('./literature_registry_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_REGISTRY_CSV_PATH = OUTPUT_DIR / 'literature_source_registry.csv'
SOURCE_REGISTRY_JSON_PATH = OUTPUT_DIR / 'literature_source_registry_combined.json'
FORMATTED_REFERENCES_MD_PATH = OUTPUT_DIR / 'formatted_reference_list.md'

AI_BACKEND = 'none'
AI_BASE_URL = 'http://127.0.0.1:11434/v1'
AI_MODEL = 'llama3.1:8b'
AI_API_KEY = None

display(pd.DataFrame([
    {'registry': 'Disease', 'path': str(DISEASE_SOURCE_JSON_PATH), 'n_sources': len(DISEASE_SOURCES)},
    {'registry': 'ROI', 'path': str(ROI_SOURCE_JSON_PATH), 'n_sources': len(ROI_SOURCES)},
    {'registry': 'manifest', 'path': str(SOURCE_REGISTRY_MANIFEST_PATH), 'n_sources': len(LITERATURE_SOURCES)},
]))


In [ ]:
def build_source_registry_df(sources):
    rows = []
    for idx, item in enumerate(sources, start=1):
        input_value = str(item.get('input_value', '')).strip()
        if input_value.startswith('http://') or input_value.startswith('https://'):
            source_type = 'web_url'
        elif input_value.lower().endswith('.pdf'):
            source_type = 'local_pdf'
        else:
            source_type = 'other'
        rows.append({
            'registry_id': idx,
            'paper_code': item.get('paper_code') or f'SRC-{idx:03d}',
            'source_label': item.get('source_label'),
            'input_value': input_value,
            'source_type': source_type,
            'source_group': item.get('source_group'),
            'evidence_category': item.get('evidence_category'),
            'disease_focus': item.get('disease_focus'),
            'evidence_use': item.get('evidence_use'),
            'note': item.get('note'),
        })
    return pd.DataFrame(rows)


source_registry_df = build_source_registry_df(LITERATURE_SOURCES)
source_registry_df.to_csv(SOURCE_REGISTRY_CSV_PATH, index=False)
SOURCE_REGISTRY_JSON_PATH.write_text(
    json.dumps(source_registry_df.to_dict(orient='records'), indent=2, ensure_ascii=False),
    encoding='utf-8',
)

display(Markdown('## Current Registered Source Inputs'))
display(source_registry_df)

display(Markdown('### Registry category counts'))
display(source_registry_df.groupby('evidence_category', dropna=False).size().reset_index(name='n_sources'))


## Stage 2. Top-Level Paper Visibility

This section should stay near the top of the workflow so that the current included papers are always visible.


In [ ]:
display(Markdown('## Current Included Papers (Input Registry View)'))
display(source_registry_df[[
    'registry_id', 'paper_code', 'source_label', 'input_value',
    'evidence_category', 'source_group', 'disease_focus', 'evidence_use', 'note'
]])


## Stage 3. Build Database From Registered Sources

This step uses the source registry as the single source of truth.

Generated outputs:

- `literature_database.sql`
- `literature_database.sqlite`
- `literature_database.json`
- `literature_database.xlsx`


In [ ]:
INPUTS = source_registry_df['input_value'].dropna().astype(str).tolist()

build_result = analyze_sources(
    inputs=INPUTS,
    output_dir=OUTPUT_DIR,
    ai_backend=AI_BACKEND,
    ai_base_url=AI_BASE_URL,
    ai_model=AI_MODEL,
    ai_api_key=AI_API_KEY,
)

records_df = pd.DataFrame(build_result['records'])
if len(records_df) > 0:
    records_df = records_df.copy()
    records_df['paper_id'] = range(1, len(records_df) + 1)
    records_df['source_title'] = records_df['title']

display(Markdown('### Database build result'))
display(pd.DataFrame([
    {'item': 'n_registered_sources', 'value': len(INPUTS)},
    {'item': 'n_extracted_records', 'value': len(build_result['records'])},
    {'item': 'sql_path', 'value': build_result.get('sql_path')},
    {'item': 'sqlite_path', 'value': build_result.get('sqlite_path')},
    {'item': 'json_path', 'value': build_result.get('json_path')},
    {'item': 'xlsx_path', 'value': build_result.get('xlsx_path')},
]))


## Stage 4. Current Included Papers (Extracted Record View)

This is the table you can show at the top of the workflow when you want title / DOI / paper ID visibility.


In [ ]:
display(Markdown('## Current Included Papers (Extracted Record View)'))
paper_preview_cols = [
    c for c in [
        'paper_id',
        'source_label',
        'source_title',
        'year',
        'doi',
        'study_design',
        'diagnoses',
        'rois',
        'imaging_metrics',
        'source_path_or_url',
    ]
    if c in records_df.columns
]
display(records_df[paper_preview_cols])


## Stage 5. Evidence Linkage Tables

This section separates two evidence types:

1. **Disease evidence**
   - disease labels, clinical/pathological features, title, DOI, paper code, and source detail URL
2. **ROI / imaging evidence**
   - ROI, imaging metric, diagnosis label, title, DOI, paper code, source detail URL, and matching rule

Every evidence-linked row should clearly show:

- `paper_code`
- `paper_id`
- `source_title`
- `doi`
- `source_detail_url`

This lets an evidence match link back to the source registry / full paper detail.


In [ ]:
evidence_df = build_evidence_framework(build_result['records'])
disease_evidence_df = build_pathology_evidence_table(build_result['records'])
roi_evidence_df = build_roi_evidence_table(build_result['records'])

if len(records_df) > 0:
    records_df = records_df.copy()
    if 'paper_code' not in records_df.columns:
        records_df['paper_code'] = records_df['paper_id'].apply(lambda x: f'P{int(x):03d}')
    merged_df = source_registry_df.merge(
        records_df,
        left_on='input_value',
        right_on='source_path_or_url',
        how='left'
    )
else:
    merged_df = source_registry_df.copy()

if 'paper_code' not in merged_df.columns and 'paper_id' in merged_df.columns:
    merged_df['paper_code'] = merged_df['paper_id'].apply(lambda x: f'P{int(x):03d}' if pd.notna(x) else None)

display(Markdown('### Source-to-record linkage table'))
display(merged_df[[
    c for c in [
        'registry_id',
        'paper_code',
        'paper_id',
        'source_label',
        'source_title',
        'doi',
        'input_value',
        'source_path_or_url',
        'disease_focus',
        'diagnoses',
        'rois',
        'imaging_metrics',
        'note',
    ]
    if c in merged_df.columns
]])

display(Markdown('### Disease evidence table'))
pathology_cols = [c for c in [
    'paper_code', 'paper_id', 'source_title', 'doi', 'source_detail_url',
    'diagnosis', 'pathology_or_clinical_feature', 'study_design', 'note'
] if c in disease_evidence_df.columns]
display(disease_evidence_df[pathology_cols].head(80))

display(Markdown('### ROI / imaging evidence table'))
roi_cols = [c for c in [
    'paper_code', 'paper_id', 'source_title', 'doi', 'source_detail_url',
    'diagnosis', 'roi_name', 'imaging_metric', 'modalities',
    'matching_rule', 'manual_review_status', 'note'
] if c in roi_evidence_df.columns]
display(roi_evidence_df[roi_cols].head(80))

display(Markdown('### Raw evidence framework preview'))
raw_cols = [c for c in [
    'paper_code', 'paper_id', 'source_title', 'doi', 'source_detail_url',
    'evidence_category', 'diagnosis', 'roi_name', 'imaging_metric',
    'pattern_summary', 'limitations'
] if c in evidence_df.columns]
display(evidence_df[raw_cols].head(50))


## Stage 6. Formatted Reference List

This produces a readable reference list for supervision, slide decks, and appendices.


In [ ]:
def format_reference_row(row):
    authors = row.get('authors') or []
    if isinstance(authors, str):
        authors_text = authors
    else:
        authors_text = ', '.join([str(a) for a in authors if str(a).strip()])
    year = row.get('year') or 'n.d.'
    title = row.get('source_title') or row.get('title') or 'Untitled source'
    doi = row.get('doi')
    url = row.get('source_path_or_url')
    paper_id = row.get('paper_id')
    paper_code = row.get('paper_code') or (f'P{int(paper_id):03d}' if pd.notna(paper_id) else 'P---')

    parts = [f'[{paper_code}]']
    if authors_text:
        parts.append(f'{authors_text} ({year}).')
    else:
        parts.append(f'({year}).')
    parts.append(f'{title}.')
    if doi:
        parts.append(f'DOI: {doi}.')
    if url:
        parts.append(f'Source detail: {url}')
    return ' '.join(parts)


formatted_references = []
if len(records_df) > 0:
    if 'paper_code' not in records_df.columns:
        records_df['paper_code'] = records_df['paper_id'].apply(lambda x: f'P{int(x):03d}')
    formatted_references = [format_reference_row(row) for _, row in records_df.iterrows()]

FORMATTED_REFERENCES_MD_PATH.write_text(
    '# Formatted Reference List\n\n' + '\n'.join([f'- {ref}' for ref in formatted_references]),
    encoding='utf-8',
)

display(Markdown('## Formatted Reference List'))
display(Markdown('\n'.join([f'- {ref}' for ref in formatted_references]) if formatted_references else '- No references extracted yet.'))


## Stage 7. Presentation-Friendly Summary

This final summary is designed for quick reporting to a supervisor.


In [ ]:
summary_lines = [
    '# Literature Workflow Source Summary',
    '',
    f'- Number of registered source inputs: {len(source_registry_df)}',
    f"- Number of extracted paper records: {len(records_df)}",
    f"- Disease evidence rows: {len(disease_evidence_df) if 'disease_evidence_df' in globals() else 'not built'}",
    f"- ROI evidence rows: {len(roi_evidence_df) if 'roi_evidence_df' in globals() else 'not built'}",
    f"- Source registry CSV: `{SOURCE_REGISTRY_CSV_PATH}`",
    f"- Source registry JSON: `{SOURCE_REGISTRY_JSON_PATH}`",
    f"- Formatted references markdown: `{FORMATTED_REFERENCES_MD_PATH}`",
    f"- Structured database SQL: `{build_result.get('sql_path')}`",
    f"- Structured database SQLite: `{build_result.get('sqlite_path')}`",
    f"- Structured database JSON: `{build_result.get('json_path')}`",
    f"- Structured database XLSX: `{build_result.get('xlsx_path')}`",
    '',
    '## Current included papers',
]

if len(records_df) > 0:
    if 'paper_code' not in records_df.columns:
        records_df['paper_code'] = records_df['paper_id'].apply(lambda x: f'P{int(x):03d}')
    for _, row in records_df.iterrows():
        summary_lines.append(
            f"- {row.get('paper_code')} | paper_id={row.get('paper_id')} | title={row.get('source_title')} | doi={row.get('doi')} | source={row.get('source_path_or_url')}"
        )
else:
    summary_lines.append('- No extracted records yet.')

summary_lines.extend([
    '',
    '## Presentation note',
    '- Use `paper_code` in matching screenshots so each evidence row can be traced back to this source registry.',
    '- Disease evidence and ROI evidence are separated to avoid implying that every disease paper contains a regional MRI finding.',
])

display(Markdown('\n'.join(summary_lines)))
